# Par-PARSEME shared foundation on Kaggle

Runs the team's shared code: data download, scorer check, dictionary-swap baseline,
RAG index, and (with a GPU) the LLM idiom finder and idiom meanings.

**Before you start** (right-hand panel → Session options):
* *Internet*: **On** (needed to download data and models).
* *Accelerator*: **GPU T4 x2** for the LLM cells. The first part also runs on CPU.

Every cell only calls scripts in `scripts/`. Nothing is computed in the notebook itself,
so results are the same as running the scripts on any machine.

## 1. Get the code
Replace the URL with the team repo (or attach the repo as a Kaggle dataset and `cp -r` it).

In [ ]:
REPO_URL = "https://github.com/<team>/par-parseme.git"   # <-- change me
!git clone -q $REPO_URL /kaggle/working/par-parseme
%cd /kaggle/working/par-parseme
!git log -1 --oneline

# run() stops the notebook with an error when a command fails, so a later
# cell never runs on missing outputs ("!cmd" would just carry on).
import subprocess
def run(cmd):
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:  # show the output live in the notebook
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")

## 2. Install packages (about 2 minutes)
Kaggle already has PyTorch, transformers, numpy and pandas. `requirements-kaggle.txt` only adds what is missing and leaves Kaggle's versions alone. (Tested with transformers 5 and pandas 2.2: the scorer check in step 4 still gives 70.69.)

In [ ]:
run("pip install -q -r requirements-kaggle.txt")

## 3. Download the organisers' data
Clones `par-parseme/` from https://gitlab.com/parseme/sharedtask-data. Run again after the test release (10 Jan 2027).

In [ ]:
run("bash scripts/download_data.sh")

## 4. Check the scorer (5 min CPU, 1 min GPU)
Re-scores the organisers' own baseline predictions. Expected: **70.69** macro, with HE 52.54 and SL 82.93.
(Their README quotes 71.2 because their FR scores file has a known bug; all other languages match exactly.)

In [ ]:
run("python scripts/score_organizer_baseline.py")

## 5. Dictionary-swap baseline on dev (row 1)
Uses the gold idiom position. Expected: **93.26** macro (see README for why this number is so high).

In [ ]:
run("python scripts/run_dictswap.py --config configs/row1_dictswap.yaml --split dev --idiom-source gold")

## 6. Build the RAG index (1 min GPU)
Saves `outputs/rag_index/multilingual-e5-base/` and prints nearest neighbours for a few dev sentences.

In [ ]:
run("python scripts/build_rag_index.py")

## 7. LLM steps (GPU T4 x2 only)
Installs vLLM, then:

`requirements-llm.txt` installs vLLM 0.11.0 with transformers 4.57.6 (vLLM 0.11 does not work with Kaggle's transformers 5) and its own torch. pip prints dependency warnings about other Kaggle packages; they are harmless. The steps run as separate scripts, so no kernel restart is needed.

Each step checks its inputs: if one fails, the notebook stops there with an ERROR line instead of running the next cells on missing files.

1. runs the LLM idiom finder on dev and reports its accuracy against the gold idioms (member 1 owns this number);
2. writes one meaning per dev sentence (shared by all models);
3. reruns the baseline with LLM meanings, and with the LLM finder instead of gold idiom positions.

If vLLM fails to install, use `--config configs/local_cpu.yaml` style settings with `backend: transformers`
(slower, but works on any GPU).

In [ ]:
run("pip install -q -r requirements-llm.txt")

In [ ]:
run("python scripts/find_idioms.py --split dev --method llm")

In [ ]:
run("python scripts/make_meanings.py --split dev")

In [ ]:
run("python scripts/run_dictswap.py --split dev --idiom-source gold --meaning-sources meanings,train")
run("python scripts/run_dictswap.py --split dev --idiom-source file --idioms outputs/idioms/dev_llm.json --meaning-sources meanings,train")

## 8. Save results
`outputs/` and `data/meanings/` now hold predictions, scores and run_info.json files.
Download them (or *Save Version* → output as a Kaggle dataset) and commit them to the repo,
so teammates reuse them instead of recomputing.

In [ ]:
!zip -qr /kaggle/working/shared_outputs.zip outputs data/meanings && ls -lh /kaggle/working/shared_outputs.zip